In [3]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
import random
import os

ModuleNotFoundError: No module named 'tensorflow'

---- 2. SET SEED agar hasil reproducible ----

Ini penting supaya setiap kali jalan hasilnya konsisten

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

---- 3. NAMA KELAS ----

Biar confusion matrix & inference bisa tampil nama bukan angka

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


FUNGSI UTAMA - dipakai ulang untuk setiap eksperimen


In [ ]:
def run_experiment(exp_name, neurons_hidden1, neurons_hidden2,
                   epochs, batch_size):
    """
    Fungsi ini menjalankan 1 eksperimen lengkap.
    Parameter yang bisa diubah:
    - neurons_hidden1 : jumlah neuron di hidden layer pertama
    - neurons_hidden2 : jumlah neuron di hidden layer kedua
    - epochs          : berapa kali model melihat seluruh data
    - batch_size      : berapa data per update gradient
    """

    print(f"\n{'='*60}")
    print(f"  EKSPERIMEN: {exp_name}")
    print(f"  Arsitektur: Dense({neurons_hidden1}) -> Dense({neurons_hidden2}) -> Dense(10)")
    print(f"  Epochs: {epochs} | Batch Size: {batch_size}")
    print(f"{'='*60}\n")

    # ---- DATA PREPROCESSING ----
    # Load dataset Fashion-MNIST
    (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

    # Normalisasi: ubah nilai pixel dari [0,255] → [0,1]
    # Ini membantu gradient descent konvergen lebih cepat
    x_train = x_train / 255.0
    x_test  = x_test  / 255.0

    # Flatten: ubah gambar 28x28 → vektor 784
    # Karena kita pakai Dense layer (bukan CNN), inputnya harus 1D
    x_train = x_train.reshape(-1, 784)
    x_test  = x_test.reshape(-1, 784)

    # One-hot encoding: ubah label angka → vektor biner
    # Contoh: label 3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
    y_train_cat = to_categorical(y_train, 10)
    y_test_cat  = to_categorical(y_test,  10)

    # ---- ARSITEKTUR MODEL ----
    # Sequential artinya layer disusun berurutan
    # Dense = fully connected layer (setiap neuron terhubung ke semua neuron layer sebelumnya)
    # ReLU  = fungsi aktivasi, membuat nilai negatif jadi 0 (mengatasi vanishing gradient)
    # Softmax = untuk output klasifikasi multi-kelas, mengubah output jadi probabilitas (total = 1)
    model = models.Sequential([
        layers.Dense(neurons_hidden1, activation='relu', input_shape=(784,)),
        layers.Dense(neurons_hidden2, activation='relu'),
        layers.Dense(10, activation='softmax')   # 10 kelas output
    ])

    # ---- COMPILE ----
    # SGD (Stochastic Gradient Descent) dengan momentum:
    # - learning_rate: seberapa besar langkah update bobot
    # - momentum: membantu "mendorong" agar tidak terjebak di local minima
    # categorical_crossentropy: loss function untuk klasifikasi multi-kelas dengan one-hot
    optimizer = optimizers.SGD(learning_rate=0.01, momentum=0.9)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()

    # Tambahkan di bagian paling atas setelah import
    import os
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

    # ---- TRAINING ----
    # validation_split=0.2 artinya 20% data training dipakai untuk validasi
    # Validasi digunakan untuk deteksi overfitting — model tidak belajar dari data ini
    history = model.fit(
        x_train, y_train_cat,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        verbose=1
    )

    # ---- EVALUASI ----
    test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
    print(f"\n[{exp_name}] Test Loss    : {test_loss:.4f}")
    print(f"[{exp_name}] Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

    # ---- VISUALISASI ACCURACY & LOSS ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{exp_name} - Accuracy & Loss', fontsize=14, fontweight='bold')

    # Accuracy plot
    # Kalau val_accuracy jauh di bawah accuracy → overfitting!
    axes[0].plot(history.history['accuracy'],     label='Train Accuracy',      color='royalblue')
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='darkorange')
    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss plot
    # Kalau val_loss mulai naik sementara train_loss turun → overfitting!
    axes[1].plot(history.history['loss'],     label='Train Loss',      color='royalblue')
    axes[1].plot(history.history['val_loss'], label='Validation Loss', color='darkorange')
    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ---- CONFUSION MATRIX ----
    # Confusion matrix menunjukkan: baris = kelas sebenarnya, kolom = kelas prediksi
    # Diagonal utama = prediksi benar, selain diagonal = prediksi salah
    y_pred       = model.predict(x_test, verbose=0)
    y_pred_class = np.argmax(y_pred, axis=1)   # ambil indeks probabilitas tertinggi

    cm = confusion_matrix(y_test, y_pred_class)

    plt.figure(figsize=(12, 9))
    sns.heatmap(cm,
                annot=True, fmt='d',
                xticklabels=class_names,
                yticklabels=class_names,
                cmap='Blues')
    plt.title(f'{exp_name} - Confusion Matrix', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Classification report: precision, recall, f1-score per kelas
    print(f"\nClassification Report [{exp_name}]:\n")
    print(classification_report(y_test, y_pred_class, target_names=class_names))

    # ---- INFERENCE (Prediksi Sampel) ----
    # Pilih 10 gambar acak dari test set dan tampilkan hasil prediksinya
    np.random.seed(SEED)
    indices = np.random.choice(len(x_test), 10, replace=False)

    plt.figure(figsize=(15, 6))
    plt.suptitle(f'{exp_name} - Inference (10 Random Samples)', fontsize=13, fontweight='bold')

    for i, idx in enumerate(indices):
        image        = x_test[idx].reshape(28, 28)           # kembalikan ke 28x28 untuk ditampilkan
        pred_proba   = model.predict(x_test[idx].reshape(1, 784), verbose=0)
        pred_label   = np.argmax(pred_proba)
        true_label   = y_test[idx]
        confidence   = pred_proba[0][pred_label] * 100

        plt.subplot(2, 5, i + 1)
        plt.imshow(image, cmap='gray')

        # Warna judul: hijau kalau benar, merah kalau salah
        color = 'green' if pred_label == true_label else 'red'
        plt.title(
            f"Pred: {class_names[pred_label]}\nTrue: {class_names[true_label]}\n{confidence:.1f}%",
            color=color, fontsize=7
        )
        plt.axis('off')

    plt.tight_layout()
    plt.show()

    return test_acc

: 

JALANKAN 3 EKSPERIMEN

In [ ]:
# ---- MODEL AWAL ----
# Arsitektur dasar mengikuti contoh MNIST di modul
# Hidden layer: 128 → 64 neuron, epoch=10, batch=64
acc_baseline = run_experiment(
    exp_name        = "Model Awal",
    neurons_hidden1 = 128,
    neurons_hidden2 = 64,
    epochs          = 10,
    batch_size      = 64
)

# ---- EKSPERIMEN 1: Ubah jumlah neuron ----
# Tambah kapasitas model dengan neuron lebih banyak
# Hipotesis: neuron lebih banyak → bisa tangkap pola lebih kompleks
# Risiko: kalau terlalu banyak & data kecil → overfit
acc_exp1 = run_experiment(
    exp_name        = "Eksperimen 1 - Neuron lebih banyak",
    neurons_hidden1 = 256,    # 2x lipat dari model awal
    neurons_hidden2 = 128,    # 2x lipat dari model awal
    epochs          = 10,
    batch_size      = 64
)

# ---- EKSPERIMEN 2: Ubah batch size ----
# Batch size kecil = update lebih sering, gradient lebih noisy tapi bisa escape local minima
# Batch size 32 (lebih kecil dari 64) → update lebih sering per epoch
# Perhatikan: training lebih lambat tapi bisa lebih stabil
acc_exp2 = run_experiment(
    exp_name        = "Eksperimen 2 - Batch Size 32",
    neurons_hidden1 = 128,
    neurons_hidden2 = 64,
    epochs          = 10,
    batch_size      = 32    # lebih kecil dari baseline
)


# =============================================================
# TABEL PERBANDINGAN HASIL
# =============================================================
print("\n" + "="*75)
print("  TABEL PERBANDINGAN EKSPERIMEN")
print("="*75)
print(f"{'No':<4} {'Arsitektur':<35} {'Epoch':<8} {'Batch':<8} {'Test Acc':<12} {'Catatan'}")
print("-"*75)
print(f"{'1':<4} {'Dense(128)->Dense(64)->Dense(10)':<35} {'10':<8} {'64':<8} {acc_baseline*100:.2f}%       Model Awal (baseline)")
print(f"{'2':<4} {'Dense(256)->Dense(128)->Dense(10)':<35} {'10':<8} {'64':<8} {acc_exp1*100:.2f}%       Neuron 2x lebih banyak")
print(f"{'3':<4} {'Dense(128)->Dense(64)->Dense(10)':<35} {'10':<8} {'32':<8} {acc_exp2*100:.2f}%       Batch size lebih kecil")
print("="*75)


  EKSPERIMEN: Model Awal
  Arsitektur: Dense(128) -> Dense(64) -> Dense(10)
  Epochs: 10 | Batch Size: 64



c:\Users\hafid\anaconda3\envs\mlenv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7871 - loss: 0.5986 - val_accuracy: 0.8422 - val_loss: 0.4425
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8484 - loss: 0.4179 - val_accuracy: 0.8584 - val_loss: 0.3979
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8648 - loss: 0.3741 - val_accuracy: 0.8609 - val_loss: 0.3874
Epoch 4/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8735 - loss: 0.3486 - val_accuracy: 0.8663 - val_loss: 0.3706
Epoch 5/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8799 - loss: 0.3297 - val_accuracy: 0.8726 - val_loss: 0.3613
Epoch 6/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8853 - loss: 0.3133 - val_accuracy: 0.8746 - val_loss: 0.3511
Epoch 7/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8898 - loss: 0.3005 - val_accuracy: 0.8747 - val_loss: 0.3565
Epoch 8/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8940 - loss: 0.2888 - val_accuracy: 0.